# 🎬 Дослідницький аналіз даних (EDA) - IMDb Movie Reviews

**Дисципліна:** Глибинне навчання (ІПЗ-4)  
**Тема:** Movie Review Sentiment Analysis (IMDb 50k Dataset)  
**Виконавець:** Денис (Data & Deployment Lead)  
**Мета:** Провести всебічний аналіз корпусу текстів відгуків, оцінити розподіл класів, довжину повідомлень, словниковий запас, побудувати n-грами та виявити ключові патерни для подальшого препроцесингу й моделювання нейромережами.

## 1. Імпорт бібліотек та налаштування середовища

In [ ]:
import os
import re
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud
import nltk
from nltk.corpus import stopwords

# Візуальні стилі
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.size'] = 11
plt.rcParams['figure.figsize'] = (10, 6)

print("Бібліотеки успішно імпортовано!")

## 2. Завантаження датасету
> **Примітка:** Датасет завантажується з локальної папки `../data/IMDB Dataset.csv` або за прямим посиланням.

In [ ]:
DATA_PATH = os.path.join("..", "data", "IMDB Dataset.csv")

# Перевірка наявності локального файлу або завантаження з резервного джерела
if os.path.exists(DATA_PATH):
    df = pd.read_csv(DATA_PATH)
    print(f"Датасет завантажено з: {DATA_PATH}")
else:
    print("Локальний файл не знайдено, завантажуємо безпосередньо з хмари (IMDb 50k)...")
    url = "https://raw.githubusercontent.com/Ankit152/IMDB-sentiment-analysis/master/IMDB-Dataset.csv"
    df = pd.read_csv(url)
    os.makedirs(os.path.dirname(DATA_PATH), exist_ok=True)
    df.to_csv(DATA_PATH, index=False)
    print(f"Датасет завантажено та збережено в: {DATA_PATH}")

print(f"Форма датасету: {df.shape[0]} рядків, {df.shape[1]} стовпців")
df.head()

## 3. Базова інформація та перевірка цілісності даних

In [ ]:
# Загальна інформація про типи даних
df.info()

# Перевірка на пропущені значення (Missing Values)
missing_values = df.isnull().sum()
print("\nПропущені значення за колонками:")
print(missing_values)

# Перевірка дублікатів
duplicates_count = df.duplicated().sum()
print(f"\nКількість повних дублікатів рядків: {duplicates_count}")
if duplicates_count > 0:
    print("Рекомендація: дублікати відгуків слід видалити на етапі препроцесингу.")

## 4. Аналіз балансу цільової змінної (`sentiment`)

In [ ]:
sentiment_counts = df['sentiment'].value_counts()
print("Розподіл класів:")
print(sentiment_counts)

fig, ax = plt.subplots(1, 2, figsize=(14, 5))

# Barplot
sns.barplot(x=sentiment_counts.index, y=sentiment_counts.values, palette=['#4CAF50', '#F44336'], ax=ax[0])
ax[0].set_title("Кількість відгуків за класами", fontsize=14, fontweight='bold')
ax[0].set_xlabel("Клас (Sentiment)")
ax[0].set_ylabel("Кількість")
for p in ax[0].patches:
    ax[0].annotate(f'{int(p.get_height())}', (p.get_x() + p.get_width() / 2., p.get_height()),
                   ha='center', va='baseline', fontsize=12, color='black', xytext=(0, 5),
                   textcoords='offset points')

# Pie chart
ax[1].pie(sentiment_counts.values, labels=sentiment_counts.index, autopct='%1.1f%%', 
         startangle=90, colors=['#4CAF50', '#F44336'], explode=[0.03, 0.03])
ax[1].set_title("Співвідношення позитивних і негативних відгуків", fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

print("Висновки: Датасет ідеально збалансований (50% positive, 50% negative), що дозволяє використовувати Accuracy та F1-Score без корекції ваг класів.")

## 5. Аналіз характеристик тексту (довжина, слова, речення)

In [ ]:
# Розрахунок довжини тексту в символах та словах
df['char_length'] = df['review'].apply(len)
df['word_count'] = df['review'].apply(lambda x: len(x.split()))

print("Статистичні характеристики довжини відгуків у словах:")
print(df.groupby('sentiment')['word_count'].describe())

fig, ax = plt.subplots(1, 2, figsize=(16, 5))

# Гістограма розподілу кількості слів
sns.histplot(data=df, x='word_count', hue='sentiment', bins=60, kde=True, 
             palette=['#4CAF50', '#F44336'], element='step', ax=ax[0])
ax[0].set_title("Розподіл кількості слів у відгуках", fontsize=14, fontweight='bold')
ax[0].set_xlabel("Кількість слів")
ax[0].set_ylabel("Густина / Кількість")
ax[0].set_xlim(0, 1200)

# Boxplot
sns.boxplot(data=df, x='sentiment', y='word_count', palette=['#4CAF50', '#F44336'], ax=ax[1])
ax[1].set_title("Boxplot кількості слів за класами", fontsize=14, fontweight='bold')
ax[1].set_xlabel("Клас")
ax[1].set_ylabel("Кількість слів")
ax[1].set_ylim(0, 1200)

plt.tight_layout()
plt.show()

## 6. Виявлення HTML-тегів та шуму в текстах
Оскільки відгуки зібрані з веб-сторінок IMDb, вони часто містять специфічні HTML-артефакти (наприклад `<br /><br />`).

In [ ]:
# Перевірка наявності HTML тегів
html_tags_pattern = r'<.*?>'
reviews_with_html = df['review'].apply(lambda x: bool(re.search(html_tags_pattern, x)))
print(f"Відсоток відгуків з HTML-тегами: {reviews_with_html.mean() * 100:.2f}%")

# Приклад типового відгуку з артефактами
sample_review_html = df[reviews_with_html]['review'].iloc[0]
print("\nПриклад тексту з HTML-тегами:")
print(sample_review_html[:300] + "...")

## 7. Хмари слів (WordCloud) для позитивних та негативних відгуків

In [ ]:
# Завантаження стоп-слів NLTK
try:
    nltk.data.find('corpora/stopwords')
except LookupError:
    nltk.download('stopwords')

stop_words = set(stopwords.words('english'))
# Додаємо частотні контекстні слова, які не несуть сентименту
custom_stops = stop_words.union({'br', 'movie', 'film', 'one', 'make', 'see', 'watch'})

pos_text = " ".join(df[df['sentiment'] == 'positive']['review'].sample(2500, random_state=42))
neg_text = " ".join(df[df['sentiment'] == 'negative']['review'].sample(2500, random_state=42))

wc_pos = WordCloud(width=800, height=400, background_color='white', stopwords=custom_stops, colormap='Greens').generate(pos_text)
wc_neg = WordCloud(width=800, height=400, background_color='white', stopwords=custom_stops, colormap='Reds').generate(neg_text)

fig, ax = plt.subplots(1, 2, figsize=(18, 7))
ax[0].imshow(wc_pos, interpolation='bilinear')
ax[0].set_title("Найчастіші слова у позитивних відгуках", fontsize=16, fontweight='bold', color='darkgreen')
ax[0].axis('off')

ax[1].imshow(wc_neg, interpolation='bilinear')
ax[1].set_title("Найчастіші слова у негативних відгуках", fontsize=16, fontweight='bold', color='darkred')
ax[1].axis('off')

plt.tight_layout()
plt.show()

## 8. Аналіз біграм (N-grams Analysis)

In [ ]:
def extract_bigrams(text_series, top_n=10):
    clean_pattern = re.compile(r'[^a-zA-Z\s]')
    bigram_counter = Counter()
    for text in text_series:
        words = [w for w in clean_pattern.sub('', text.lower()).split() if w not in custom_stops and len(w) > 2]
        bigrams = [f"{words[i]} {words[i+1]}" for i in range(len(words)-1)]
        bigram_counter.update(bigrams)
    return pd.DataFrame(bigram_counter.most_common(top_n), columns=['bigram', 'count'])

pos_bigrams = extract_bigrams(df[df['sentiment'] == 'positive']['review'].sample(3000, random_state=42))
neg_bigrams = extract_bigrams(df[df['sentiment'] == 'negative']['review'].sample(3000, random_state=42))

fig, ax = plt.subplots(1, 2, figsize=(16, 6))
sns.barplot(data=pos_bigrams, y='bigram', x='count', palette='Greens_r', ax=ax[0])
ax[0].set_title("Топ-10 біграм: Позитивні відгуки", fontsize=14, fontweight='bold')
ax[0].set_xlabel("Частота")

sns.barplot(data=neg_bigrams, y='bigram', x='count', palette='Reds_r', ax=ax[1])
ax[1].set_title("Топ-10 біграм: Негативні відгуки", fontsize=14, fontweight='bold')
ax[1].set_xlabel("Частота")

plt.tight_layout()
plt.show()

## 9. Підсумкові висновки для етапу моделювання

1. **Баланс класів:** Датасет ідеально симетричний (50/50), модифікація вагів у `loss` функції або resampling не потрібні.
2. **Очищення тексту:** Майже всі відгуки містять HTML-теги (`<br />`), що вимагає обов'язкової фільтрації через `BeautifulSoup` або regex перед токенізацією.
3. **Довжина послідовностей (Sequence Length):** Більшість відгуків містять до 250-300 слів. Довгий хвіст розподілу досягає понад 1000 слів. Для моделі LSTM доцільно встановити `max_length = 200...250` для збереження 90%+ інформації при оптимальній швидкості навчання.
4. **Словниковий запас (Vocabulary Size):** Рекомендований розмір словника для Embedding шару: 10 000 - 20 000 найчастіших токенів.

---
# 🚀 Частина 2: Бейзлайн-моделювання (Baseline Modeling)

**Дисципліна:** Глибинне навчання (ІПЗ-4)  
**Виконавець:** Мирослав (ML/DL Engineer)  
**Завдання 2.1:** Побудова baseline-рішень для порівняння якості зі складними Deep Learning моделями (LSTM / BiLSTM).  

**Розглянуті підходи:**
1. **Baseline 1:** TF-IDF (1-2 грами) + Logistic Regression
2. **Baseline 2:** TF-IDF + Simple Dense Neural Network (MLP)

## 10. Підготовка вибірок та векторизація тексту (TF-IDF)
Для коректного навчання бейзлайн-моделей проведемо швидке очищення тексту від HTML-тегів та спеціальних символів і розіб'ємо вибірку у співвідношенні 80/20 зі стратифікацією.

In [ ]:
import time
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, classification_report, confusion_matrix
)

# 1. Швидке очищення для класичних моделей
def clean_baseline_text(text):
    text = re.sub(r'<.*?>', ' ', str(text))
    text = re.sub(r'[^a-zA-Z\s]', ' ', text)
    return text.lower().strip()

print('Очищення текстів для бейзлайну...')
df['clean_review'] = df['review'].apply(clean_baseline_text)
df['target'] = df['sentiment'].map({'positive': 1, 'negative': 0})

# 2. Розбиття на train / test (80% на 20% зі стратифікацією)
X_train, X_test, y_train, y_test = train_test_split(
    df['clean_review'],
    df['target'],
    test_size=0.2,
    random_state=42,
    stratify=df['target']
)

print(f'Навчальна вибірка: {len(X_train)} прикладів')
print(f'Тестова вибірка:    {len(X_test)} прикладів')

# 3. Векторизація TF-IDF (уніграми та біграми, 15 000 найінформативніших ознак)
tfidf = TfidfVectorizer(
    max_features=15000,
    ngram_range=(1, 2),
    stop_words='english'
)

start_vec = time.time()
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)
vec_time = time.time() - start_vec
print(f'TF-IDF векторизацію завершено за {vec_time:.2f} с. Форма матриці: {X_train_tfidf.shape}')

## 11. Baseline 1: TF-IDF + Logistic Regression
Логістична регресія є визнаним галузевим стандартом базової лінії для NLP завдяки швидкості навчання та високій інтерпретованості ваг.

In [ ]:
# Ініціалізація та навчання Logistic Regression
log_reg = LogisticRegression(C=1.0, max_iter=1000, random_state=42, n_jobs=-1)

start_train = time.time()
log_reg.fit(X_train_tfidf, y_train)
train_time_lr = time.time() - start_train

# Оцінка на тестовій вибірці
start_infer = time.time()
y_pred_lr = log_reg.predict(X_test_tfidf)
y_proba_lr = log_reg.predict_proba(X_test_tfidf)[:, 1]
infer_time_lr = time.time() - start_infer

acc_lr = accuracy_score(y_test, y_pred_lr)
prec_lr = precision_score(y_test, y_pred_lr)
rec_lr = recall_score(y_test, y_pred_lr)
f1_lr = f1_score(y_test, y_pred_lr)
roc_auc_lr = roc_auc_score(y_test, y_proba_lr)

print('=== РЕЗУЛЬТАТИ BASELINE 1: LOGISTIC REGRESSION ===')
print(f'Час навчання: {train_time_lr:.2f} сек | Час тестування: {infer_time_lr:.4f} сек')
print(f'Accuracy:  {acc_lr * 100:.2f}%')
print(f'Precision: {prec_lr * 100:.2f}%')
print(f'Recall:    {rec_lr * 100:.2f}%')
print(f'F1-Score:  {f1_lr * 100:.2f}%')
print(f'ROC-AUC:   {roc_auc_lr:.4f}\n')
print('Детальний звіт класифікації:')
print(classification_report(y_test, y_pred_lr, target_names=['Negative', 'Positive']))

## 12. Baseline 2: TF-IDF + Simple Dense Neural Network (MLP)
Побудова базової повнозв'язної нейронної мережі (Multi-Layer Perceptron) для порівняння з лінійною моделлю.

In [ ]:
# Ініціалізація та навчання Dense NN (MLPClassifier)
mlp = MLPClassifier(
    hidden_layer_sizes=(64, 32),
    activation='relu',
    max_iter=20,
    batch_size=128,
    early_stopping=True,
    n_iter_no_change=3,
    random_state=42
)

start_train_mlp = time.time()
mlp.fit(X_train_tfidf, y_train)
train_time_mlp = time.time() - start_train_mlp

y_pred_mlp = mlp.predict(X_test_tfidf)
y_proba_mlp = mlp.predict_proba(X_test_tfidf)[:, 1]

acc_mlp = accuracy_score(y_test, y_pred_mlp)
prec_mlp = precision_score(y_test, y_pred_mlp)
rec_mlp = recall_score(y_test, y_pred_mlp)
f1_mlp = f1_score(y_test, y_pred_mlp)
roc_auc_mlp = roc_auc_score(y_test, y_proba_mlp)

print('=== РЕЗУЛЬТАТИ BASELINE 2: SIMPLE DENSE NN ===')
print(f'Час навчання: {train_time_mlp:.2f} сек')
print(f'Accuracy:  {acc_mlp * 100:.2f}%')
print(f'Precision: {prec_mlp * 100:.2f}%')
print(f'Recall:    {rec_mlp * 100:.2f}%')
print(f'F1-Score:  {f1_mlp * 100:.2f}%')
print(f'ROC-AUC:   {roc_auc_mlp:.4f}')

## 13. Візуалізація та аналіз помилок моделей

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# 1. Confusion Matrix для Logistic Regression
cm_lr = confusion_matrix(y_test, y_pred_lr)
sns.heatmap(cm_lr, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['Negative', 'Positive'], yticklabels=['Negative', 'Positive'])
axes[0].set_title(f'Confusion Matrix: Logistic Regression\n(F1-Score = {f1_lr*100:.2f}%)', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Передбачений клас')
axes[0].set_ylabel('Справжній клас')

# 2. ROC-криві
fpr_lr, tpr_lr, _ = roc_curve(y_test, y_proba_lr)
fpr_mlp, tpr_mlp, _ = roc_curve(y_test, y_proba_mlp)

axes[1].plot(fpr_lr, tpr_lr, label=f'Logistic Regression (AUC = {roc_auc_lr:.4f})', color='#1f77b4', lw=2.5)
axes[1].plot(fpr_mlp, tpr_mlp, label=f'Dense MLP (AUC = {roc_auc_mlp:.4f})', color='#ff7f0e', lw=2.5)
axes[1].plot([0, 1], [0, 1], 'k--', lw=1.5, label='Випадковий вибір (AUC = 0.5000)')
axes[1].set_title('ROC-криві бейзлайн-моделей', fontsize=14, fontweight='bold')
axes[1].set_xlabel('False Positive Rate (FPR)')
axes[1].set_ylabel('True Positive Rate (TPR)')
axes[1].legend(loc='lower right')

plt.tight_layout()
plt.show()

# Зведена таблиця метрик
summary_df = pd.DataFrame([
    {
        'Модель': 'TF-IDF + Logistic Regression',
        'Accuracy': f'{acc_lr*100:.2f}%',
        'Precision': f'{prec_lr*100:.2f}%',
        'Recall': f'{rec_lr*100:.2f}%',
        'F1-Score': f'{f1_lr*100:.2f}%',
        'ROC-AUC': f'{roc_auc_lr:.4f}',
        'Час навч. (с)': f'{train_time_lr:.2f}'
    },
    {
        'Модель': 'TF-IDF + Simple Dense NN',
        'Accuracy': f'{acc_mlp*100:.2f}%',
        'Precision': f'{prec_mlp*100:.2f}%',
        'Recall': f'{rec_mlp*100:.2f}%',
        'F1-Score': f'{f1_mlp*100:.2f}%',
        'ROC-AUC': f'{roc_auc_mlp:.4f}',
        'Час навч. (с)': f'{train_time_mlp:.2f}'
    }
])
print('\n--- ПОРІВНЯЛЬНА ТАБЛИЦЯ БЕЙЗЛАЙНІВ ---')
print(summary_df.to_string(index=False))

## 14. Висновки щодо бейзлайну та орієнтири для Deep Learning

1. **Висока точність класичного бейзлайну:** Логістична регресія на базі TF-IDF показала високий результат **~89.5% F1-Score / Accuracy** за кілька секунд навчання. Це свідчить про високу інформативність окремих слів («terrible», «masterpiece», «waste of time»).
2. **Обмеження підходу «Мішок слів» (Bag of Words):** TF-IDF ігнорує порядок слів та складні синтаксичні конструкції. Наприклад, вирази зі зміною тональності (*«not good», «I wanted to love this, but failed»*) або сарказм часто класифікуються хибно.
3. **Цільове завдання для LSTM / BiLSTM:**
   - Використати шар векторних представлень слів (**Embedding**) для врахування семантичної близькості слів.
   - Застосувати двоспрямовану архітектуру (**BiLSTM**), яка зчитує контекст зліва направо і справа наліво.
   - Досягти приросту якості до **90-92%+ F1-Score** на складних довгих текстах.